In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.interpolate import interp1d

# --- Configuración ---
RUTA_ARCHIVO_ENTRADA = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/dlocal_ratios trimestrales.xlsx')
NOMBRE_HOJA_ENTRADA = 'ratios trimestrales'

DIRECTORIO_SALIDA = Path('C:/Users/Andy/OneDrive/Desktop/MCD/Tesis/Datos_fuente_Bloomberg/en valores/serie completa 2014-2024/Dataset/Interpolado_EmpresaUnica/')
PLANTILLA_NOMBRE_ARCHIVO_LINEAL = '{nombre_empresa}_mensual_lineal.xlsx'
PLANTILLA_NOMBRE_ARCHIVO_CUBICO = '{nombre_empresa}_mensual_spline_cubico.xlsx'

# --- CONFIGURACIÓN DEL DISEÑO DE TUS DATOS ---
# Esto le dice al script qué columnas son métricas a interpolar y qué columna tiene el ID de la empresa.
CONFIG_DISENO_DATOS = {
    "columna_id_empresa": "Empresa",  # Nombre de la columna con el ticker/nombre de la empresa
    "columnas_metricas": [             # Lista de nombres de columnas que son métricas financieras a interpolar
        "ROCE", "EBIT",
        "TotalActivos", "Total Activos", "Deuda a LP",
        "ROA", "Beneficio neto",
        "ROI", "EV", "Cap de mercado", "Deuda a CP", "Efectivo y equiv"
        # Agrega aquí todas las demás columnas de métricas de tu hoja Excel que quieras interpolar
    ]
    # Opcional: Si tienes una columna que etiqueta explícitamente el trimestre (ej., "2020Q1", "2020Q2")
    # podrías usarla para etiquetas mensuales más descriptivas, pero no es estrictamente necesario.
    # "columna_etiqueta_trimestral": "NombreDeColumnaConEtiquetasTrimestrales" # Por defecto es None
}

# --- Función auxiliar para generar etiquetas de índice mensuales ---
def generar_etiquetas_indice_mensual(num_periodos_trimestrales):
    """
    Genera etiquetas de índice mensuales genéricas como Q1_M3, Q2_M1, ..., QN_M3.
    Args:
        num_periodos_trimestrales (int): El número de trimestres (N).
    Returns:
        list: Una lista de etiquetas mensuales.
    """
    if num_periodos_trimestrales == 0:
        return []
    # Estos son marcadores de posición genéricos para los trimestres, ej., "T1", "T2"
    # representando el 1er, 2do trimestre *en la secuencia de datos proporcionada*.
    marcadores_trimestrales = [f"T{i+1}" for i in range(num_periodos_trimestrales)]

    if num_periodos_trimestrales == 1:
        return [f"{marcadores_trimestrales[0]}_M3"]

    etiquetas_mensuales = []
    for t in range(num_periodos_trimestrales - 1):
        etiquetas_mensuales.append(f"{marcadores_trimestrales[t]}_M3")      # Fin del trimestre actual
        etiquetas_mensuales.append(f"{marcadores_trimestrales[t+1]}_M1")    # Primer mes del siguiente trimestre
        etiquetas_mensuales.append(f"{marcadores_trimestrales[t+1]}_M2")    # Segundo mes del siguiente trimestre
    etiquetas_mensuales.append(f"{marcadores_trimestrales[num_periodos_trimestrales-1]}_M3") # Fin del último trimestre
    return etiquetas_mensuales

# --- Interpolación central para una sola columna (métrica) ---
def interpolar_columna_metrica_a_serie_mensual(
    serie_valores_trimestrales: pd.Series,
    nombre_metrica_para_log: str,
    metodo_interpolacion: str = 'linear'
) -> pd.Series:
    """
    Interpola una sola columna de datos trimestrales (una métrica para una empresa) a mensual.

    Args:
        serie_valores_trimestrales (pd.Series): Serie de valores trimestrales para la métrica.
        nombre_metrica_para_log (str): Nombre de la métrica (para mensajes de log).
        metodo_interpolacion (str): 'linear' o 'cubic'.

    Returns:
        pd.Series: Valores mensuales interpolados con un índice mensual genérico (T1_M3, T2_M1, ...).
                   Devuelve una Serie vacía si no hay datos o la interpolación falla.
    """
    # Convertir a numérico, forzando errores a NaN. Manejar marcadores no numéricos típicos.
    valores = pd.to_numeric(
        serie_valores_trimestrales.replace({'-': np.nan, '—': np.nan, '#N/A N/A': np.nan}),
        errors='coerce'
    ).values

    num_puntos_trimestrales = len(valores)  # Número de puntos de datos trimestrales para esta métrica

    if num_puntos_trimestrales == 0:
        # print(f"      Los datos para la métrica '{nombre_metrica_para_log}' están vacíos después de la conversión numérica.")
        return pd.Series(dtype=float)

    # Generar etiquetas de índice mensuales genéricas basadas en N para *esta* serie de métrica
    etiquetas_indice_mensual = generar_etiquetas_indice_mensual(num_puntos_trimestrales)

    if num_puntos_trimestrales == 1: # generar_etiquetas_indice_mensual maneja N=1
        # print(f"      Solo un punto de dato para '{nombre_metrica_para_log}'. Copiando a M3 del T1.")
        return pd.Series([valores[0]], index=etiquetas_indice_mensual)

    # `tiempos_datos_originales_en_mensual` son los índices (0, 3, 6, ...) en la *línea de tiempo mensual*
    # donde se encuentran los datos trimestrales originales.
    # La longitud de `etiquetas_indice_mensual` es el número total de puntos mensuales para este N.
    total_puntos_mensuales = len(etiquetas_indice_mensual)
    tiempos_datos_originales_en_mensual = np.arange(0, total_puntos_mensuales, 3) # Debería tener N elementos si N > 0

    # Array para contener los valores mensuales interpolados
    valores_mensuales_interpolados = np.full(total_puntos_mensuales, np.nan)

    # Encontrar índices en el array `valores` trimestrales originales que no son NaN
    indices_trimestrales_validos = np.where(~np.isnan(valores))[0]

    # Iterar sobre segmentos contiguos de datos trimestrales válidos
    for indices_segmento in np.split(indices_trimestrales_validos, np.where(np.diff(indices_trimestrales_validos) != 1)[0] + 1):
        if indices_segmento.size == 0:  # Omitir segmentos vacíos
            continue

        # `indices_segmento` son índices relativos al array `valores` original (trimestral, 0 a N-1)
        if len(indices_segmento) > 1:  # Se necesitan al menos dos puntos para interpolar
            idx_trim_inicio = indices_segmento[0]
            idx_trim_fin = indices_segmento[-1]

            # tiempos_mensuales_originales: Puntos de tiempo mensuales correspondientes a los datos trimestrales válidos en este segmento
            tiempos_mensuales_originales = tiempos_datos_originales_en_mensual[idx_trim_inicio : idx_trim_fin + 1]
            # valores_trimestrales_originales: Los valores trimestrales reales para este segmento
            valores_trimestrales_originales = valores[idx_trim_inicio : idx_trim_fin + 1]

            # tiempos_mensuales_a_interpolar: Todos los puntos de tiempo mensuales dentro del rango de este segmento
            tiempos_mensuales_a_interpolar = np.arange(tiempos_mensuales_originales[0], tiempos_mensuales_originales[-1] + 1)

            valores_interpolados_segmento = np.full(len(tiempos_mensuales_a_interpolar), np.nan)

            if metodo_interpolacion == 'linear':
                valores_interpolados_segmento = np.interp(tiempos_mensuales_a_interpolar, tiempos_mensuales_originales, valores_trimestrales_originales)
            elif metodo_interpolacion == 'cubic':
                if len(tiempos_mensuales_originales) >= 2: # interp1d necesita al menos 2 puntos
                    try:
                        # bounds_error=False y fill_value=np.nan ayudan si los puntos de interpolación están fuera del rango de datos (no debería suceder aquí)
                        funcion_interp_cubica = interp1d(tiempos_mensuales_originales, valores_trimestrales_originales, kind='cubic', bounds_error=False, fill_value=np.nan)
                        valores_interpolados_segmento = funcion_interp_cubica(tiempos_mensuales_a_interpolar)

                        # Respaldo para segmentos donde la cúbica podría tener problemas (ej., < 4 puntos) y producir NaNs
                        if np.isnan(valores_interpolados_segmento).any() and len(tiempos_mensuales_originales) < 4:
                            # print(f"      Spline cúbico para '{nombre_metrica_para_log}' (segmento con {len(tiempos_mensuales_originales)} pts) produjo NaNs. Reintentando segmento con lineal.")
                            valores_interpolados_segmento = np.interp(tiempos_mensuales_a_interpolar, tiempos_mensuales_originales, valores_trimestrales_originales)
                    except ValueError: # Falla general de la interpolación cúbica
                        # print(f"      Spline cúbico falló para '{nombre_metrica_para_log}'. Volviendo a lineal para el segmento.")
                        valores_interpolados_segmento = np.interp(tiempos_mensuales_a_interpolar, tiempos_mensuales_originales, valores_trimestrales_originales)
                else: # No hay suficientes puntos para cúbica, usar lineal para el segmento
                    valores_interpolados_segmento = np.interp(tiempos_mensuales_a_interpolar, tiempos_mensuales_originales, valores_trimestrales_originales)

            # Colocar el segmento interpolado en el array mensual completo
            valores_mensuales_interpolados[tiempos_mensuales_a_interpolar] = valores_interpolados_segmento

        elif len(indices_segmento) == 1:  # Punto de dato trimestral aislado único
            idx_trim = indices_segmento[0]
            tiempo_mensual_punto_unico = tiempos_datos_originales_en_mensual[idx_trim] # Su posición M3 en la línea de tiempo mensual
            valores_mensuales_interpolados[tiempo_mensual_punto_unico] = valores[idx_trim]

    return pd.Series(valores_mensuales_interpolados, index=etiquetas_indice_mensual)


# --- Función Principal de Procesamiento para el Diseño de Empresa Única ---
def procesar_archivo_empresa_unica(
    ruta_entrada: Path,
    nombre_hoja_entrada: str,
    configuracion: dict,
    directorio_salida: Path,
    plantilla_nombre_lineal: str,
    plantilla_nombre_cubico: str
):
    if not ruta_entrada.exists():
        raise FileNotFoundError(f"Archivo de entrada no encontrado: {ruta_entrada}")

    try:
        # Leer la hoja especificada (o la primera si nombre_hoja_entrada es None o 0)
        df_principal = pd.read_excel(ruta_entrada, sheet_name=nombre_hoja_entrada if nombre_hoja_entrada else 0)
        print(f"Hoja '{nombre_hoja_entrada if nombre_hoja_entrada else df_principal.columns[0]}' cargada exitosamente desde '{ruta_entrada}'.")
    except Exception as e:
        raise ValueError(f"No se pudo leer la hoja '{nombre_hoja_entrada}' desde '{ruta_entrada}'. Error: {e}")

    if df_principal.empty:
        print("La hoja cargada está vacía. No se realizará ningún procesamiento.")
        return

    col_id_empresa = configuracion["columna_id_empresa"]
    cols_metricas = configuracion["columnas_metricas"]

    if col_id_empresa not in df_principal.columns:
        raise ValueError(f"La columna ID de empresa '{col_id_empresa}' no se encontró en la hoja.")

    # Asumiendo que es para una sola empresa, tomar el primer ID de empresa no-NaN como su nombre
    nombre_empresa = df_principal[col_id_empresa].dropna().iloc[0] if not df_principal[col_id_empresa].dropna().empty else "EmpresaDesconocida"
    # Sanitizar el nombre de la empresa para usarlo en el nombre del archivo
    nombre_empresa_seguro = "".join(c if c.isalnum() else "_" for c in str(nombre_empresa))


    # Asegurar que el directorio de salida exista
    directorio_salida.mkdir(parents=True, exist_ok=True)

    # --- Procesar para AMBAS interpolaciones: Lineal y Cúbica ---
    for metodo_interp, plantilla_nombre_archivo in [('linear', plantilla_nombre_lineal), ('cubic', plantilla_nombre_cubico)]:
        print(f"\nProcesando para interpolación {metodo_interp.upper()} para la empresa: {nombre_empresa}")
        
        todas_metricas_interpoladas = {} # Para almacenar Series para cada métrica interpolada
        
        for nombre_columna_metrica in cols_metricas:
            if nombre_columna_metrica not in df_principal.columns:
                print(f"  Columna de métrica '{nombre_columna_metrica}' no encontrada en la hoja, omitiendo.")
                continue
            
            print(f"  Interpolando métrica: '{nombre_columna_metrica}' usando {metodo_interp}")
            datos_trimestrales_metrica = df_principal[nombre_columna_metrica]
            
            serie_interpolada = interpolar_columna_metrica_a_serie_mensual(
                datos_trimestrales_metrica,
                nombre_columna_metrica,
                metodo_interpolacion=metodo_interp
            )
            
            if not serie_interpolada.empty:
                todas_metricas_interpoladas[nombre_columna_metrica] = serie_interpolada
            else:
                print(f"    No se interpolaron datos para '{nombre_columna_metrica}'.")

        if todas_metricas_interpoladas:
            # Combinar todas las Series de métricas interpoladas en un único DataFrame
            # Pandas las alineará por su índice mensual común (T1_M3, T2_M1, etc.)
            # y rellenará con NaNs si las series tienen diferentes longitudes.
            df_salida = pd.DataFrame(todas_metricas_interpoladas)
            df_salida.index.name = "Periodo_Mensual" # Nombrar la columna del índice

            nombre_archivo_salida = plantilla_nombre_archivo.format(nombre_empresa=nombre_empresa_seguro)
            ruta_archivo_salida = directorio_salida / nombre_archivo_salida
            
            with pd.ExcelWriter(ruta_archivo_salida, engine='openpyxl') as escritor_excel:
                df_salida.to_excel(escritor_excel, sheet_name=f"{nombre_empresa_seguro}_{metodo_interp}", index=True)
            print(f"Datos interpolados ({metodo_interp.capitalize()}) para '{nombre_empresa}' guardados en: {ruta_archivo_salida}")
        else:
            print(f"No se interpolaron métricas exitosamente usando el método {metodo_interp} para '{nombre_empresa}'.")

# --- Ejecución del Script ---
if __name__ == "__main__":
    # Validar CONFIG_DISENO_DATOS
    if not CONFIG_DISENO_DATOS.get("columna_id_empresa") or not CONFIG_DISENO_DATOS.get("columnas_metricas"):
        print("Error: 'columna_id_empresa' y 'columnas_metricas' deben estar definidas en CONFIG_DISENO_DATOS.")
    else:
        try:
            procesar_archivo_empresa_unica(
                RUTA_ARCHIVO_ENTRADA,
                NOMBRE_HOJA_ENTRADA,
                CONFIG_DISENO_DATOS,
                DIRECTORIO_SALIDA,
                PLANTILLA_NOMBRE_ARCHIVO_LINEAL,
                PLANTILLA_NOMBRE_ARCHIVO_CUBICO
            )
            print("\nTodas las tareas de interpolación para la empresa única han finalizado.")
        except Exception as e:
            print(f"\nOcurrió un error durante el procesamiento: {e}")
            import traceback
            traceback.print_exc()

Hoja 'ratios trimestrales' cargada exitosamente desde 'C:\Users\Andy\OneDrive\Desktop\MCD\Tesis\Datos_fuente_Bloomberg\en valores\serie completa 2014-2024\Dataset\dlocal_ratios trimestrales.xlsx'.

Procesando para interpolación LINEAR para la empresa: DLO US Equity
  Interpolando métrica: 'ROCE' usando linear
  Interpolando métrica: 'EBIT' usando linear
  Columna de métrica 'TotalActivos' no encontrada en la hoja, omitiendo.
  Interpolando métrica: 'Total Activos' usando linear
  Interpolando métrica: 'Deuda a LP' usando linear
  Interpolando métrica: 'ROA' usando linear
  Interpolando métrica: 'Beneficio neto' usando linear
  Interpolando métrica: 'ROI' usando linear
  Interpolando métrica: 'EV' usando linear
  Interpolando métrica: 'Cap de mercado' usando linear
  Interpolando métrica: 'Deuda a CP' usando linear
  Interpolando métrica: 'Efectivo y equiv' usando linear
Datos interpolados (Linear) para 'DLO US Equity' guardados en: C:\Users\Andy\OneDrive\Desktop\MCD\Tesis\Datos_fuente_